# 原项目 vs 自采 EDF 五种对比实验（按通道数量对齐版）

这个版本不强制自采 EDF 和原项目 MAT 使用同名通道。

- 原项目 MAT：默认使用原论文的 7 个通道。
- 自采 EDF：自动选择前 7 个有效 EEG 通道，排除 A1/A2/X1/X2/X3/CM 等通道。
- 标签固定：`1 = 专注 focus`，`0 = 不专注 unfocus`。
- 自采顺序：前 10 分钟不专注，后 10 分钟专注。

注意：这种做法能保证特征维度一致，方便跑五种实验；但跨数据集比较时，通道空间位置不完全一致，报告里需要说明。

In [1]:
# =========================
# 0. 导入库
# =========================

from pathlib import Path
import warnings
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import scipy.io
from scipy import signal

import mne

from sklearn.model_selection import LeaveOneGroupOut, GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

try:
    from BaselineRemoval import BaselineRemoval
    HAS_BASELINE_REMOVAL = True
except Exception:
    HAS_BASELINE_REMOVAL = False

warnings.filterwarnings("ignore")

print("HAS_BASELINE_REMOVAL:", HAS_BASELINE_REMOVAL)

HAS_BASELINE_REMOVAL: True


In [2]:
# =========================
# 1. 参数设置
# =========================

# 你们自采 EDF 文件夹
SELF_EDF_DIR = Path(r".\newdata\formal_20min")

# 原项目 MAT 文件夹
ORIGINAL_MAT_DIR = Path(r".\originaldata")

# 统一到原项目采样率
TARGET_FS = 128

# 每个文件 20 分钟：前 10 分钟 + 后 10 分钟
TOTAL_MINUTES = 20
SEGMENT_MINUTES = 10
REQUIRED_N = TOTAL_MINUTES * 60 * TARGET_FS
SEGMENT_N = SEGMENT_MINUTES * 60 * TARGET_FS

# 标签约定：1 = 专注 focus；0 = 不专注 unfocus
LABEL_NAMES = {
    0: "unfocus",
    1: "focus",
}

# 自采数据顺序：前 10 分钟不专注，后 10 分钟专注
SELF_SEGMENT_ORDER = "unfocus_first"

# 原项目数据顺序：前 10 分钟专注，后 10 分钟不专注
ORIGINAL_SEGMENT_ORDER = "focus_first"

# 你们补充的信息：9 个文件 = 3 个人 × 每人 3 段 20 分钟
FILES_PER_SELF_SUBJECT = 3

# 如果文件排序后不是“每个人连续 3 个文件”，就手动填这个表。
# filename 必须和 EDF 文件名完全一致。
SELF_FILE_TABLE = None
# 示例：
# SELF_FILE_TABLE = pd.DataFrame([
#     {"filename": "sub01_session01.edf", "subject_id": "S01", "session_id": 1},
#     {"filename": "sub01_session02.edf", "subject_id": "S01", "session_id": 2},
#     {"filename": "sub01_session03.edf", "subject_id": "S01", "session_id": 3},
#     {"filename": "sub02_session01.edf", "subject_id": "S02", "session_id": 1},
#     {"filename": "sub02_session02.edf", "subject_id": "S02", "session_id": 2},
#     {"filename": "sub02_session03.edf", "subject_id": "S02", "session_id": 3},
#     {"filename": "sub03_session01.edf", "subject_id": "S03", "session_id": 1},
#     {"filename": "sub03_session02.edf", "subject_id": "S03", "session_id": 2},
#     {"filename": "sub03_session03.edf", "subject_id": "S03", "session_id": 3},
# ])

# 通道选择模式：
# count_only = 不强制两边使用同名通道，只保证两边“通道数量相同”。
# 这样自采 EDF 可以从自己的可用 EEG 通道里自动取 N 个，避开 A1/A2/X1/X2/X3/CM 等非脑电或参考通道。
# 注意：这种模式能跑通五种对比，但跨数据集对比不如“同名通道”严格；报告里要说明“通道数量对齐，通道位置未完全一致”。
CHANNEL_SELECTION_MODE = "count_only"

# 原论文项目原本使用 7 个通道；这里默认保持 7 个特征通道。
N_EEG_CHANNELS = 7

# 原论文项目选用的 7 个通道。原项目数据优先用这 7 个。
ORIGINAL_PAPER_CHANNELS = ["F7", "F3", "P7", "O1", "O2", "P8", "AF4"]

# 自采 EDF 自动选通道时排除这些 base 通道。
# A1/A2 多数时候是耳垂/参考；X1/X2/X3 可能是外部辅助通道；CM 通常不是正式 EEG 分析通道。
EXCLUDE_CHANNEL_BASES = {"A1", "A2", "X1", "X2", "X3", "CM"}

# 这些关键词一般也不是 EEG 主通道。
EXCLUDE_CHANNEL_KEYWORDS = ("TRIGGER", "STATUS", "STIM", "MARK", "ANNOT", "EVENT", "ECG", "EKG", "EMG")

# 原项目 MAT 里 14 个 EEG 通道的顺序
ORIGINAL_MAT_ALL_CHANNELS = [
    "AF3", "F7", "F3", "FC5", "T7", "P7", "O1",
    "O2", "P8", "T8", "FC6", "F4", "F8", "AF4",
]

# 预处理参数
LOWCUT = 0.2
HIGHCUT = 43.0
FILTER_ORDER = 5

# 原项目 STFT 参数
STFT_NPERSEG = 128
STFT_NFFT = 1024
STFT_NOVERLAP = 0

# 原项目频率 bin / 时间窗口参数
# 原项目：for i in range(1, 144, 4) -> 36 个频率 bin
FREQ_BIN_START = 1
FREQ_BIN_STOP = 144
FREQ_BIN_STEP = 4

# 原项目：每 15 个 STFT 时间窗做一次平均 -> 601 - 15 + 1 = 587? 
# 但原代码写死 585，这里采用“按原代码的 i in range(585)”以便对齐。
ORIGINAL_WINDOW_COUNT = 585
ORIGINAL_TIME_AVG_WIDTH = 15

# 模型参数
RANDOM_STATE = 42
PCA_N_COMPONENTS = 0.95
TEST_SIZE = 0.2

In [3]:
# =========================
# 2. 工具函数：通道匹配、切标签、预处理
# =========================

def simplify_channel_name(name):
    """清理 EDF 通道名，例如 'EEG F7-Pz' -> 'F7-Pz'。"""
    name = str(name).strip()
    if name.upper().startswith("EEG "):
        name = name[4:].strip()
    return name


def base_channel_name(name):
    """
    把 'F7-Pz' / 'F7-REF' / 'EEG F7-Pz' 统一提取成 'F7'。
    """
    name = simplify_channel_name(name)
    name = name.replace(" ", "")
    return re.split(r"[-_]", name)[0].upper()


def find_channel_by_base(available_channels, target_base):
    """
    在 EDF 的实际通道中，寻找和 target_base 对应的通道。
    例如 target_base='F7'，可以匹配 'F7'、'F7-Pz'、'EEG F7-Pz'。
    """
    target_base = target_base.upper()

    # 1. 完全匹配
    for ch in available_channels:
        if simplify_channel_name(ch).upper() == target_base:
            return ch

    # 2. base 匹配
    candidates = []
    for ch in available_channels:
        if base_channel_name(ch) == target_base:
            candidates.append(ch)

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1:
        # 优先选 -Pz 参考的版本
        for ch in candidates:
            if "PZ" in simplify_channel_name(ch).upper():
                return ch
        return candidates[0]

    raise ValueError(f"找不到通道 {target_base}，当前可用通道为：{available_channels}")


def is_valid_eeg_base(base):
    """
    判断一个 base 通道名是否适合作为 EEG 特征通道。
    例如 F3-Pz 的 base 是 F3；A1-Pz 的 base 是 A1，要排除。
    """
    base = str(base).upper().strip()

    if base in EXCLUDE_CHANNEL_BASES:
        return False

    # 保险：A1/A2/A数字、X1/X2/X数字 都排除。
    if re.fullmatch(r"A\d+", base):
        return False
    if re.fullmatch(r"X\d+", base):
        return False

    for kw in EXCLUDE_CHANNEL_KEYWORDS:
        if kw in base:
            return False

    # 常见 10-20 / 10-10 EEG 命名：Fp1, F3, Fz, Cz, P4, O2, T5, FC5 等。
    return re.fullmatch(r"(FP|AF|F|FT|FC|C|CP|TP|T|P|PO|O)[0-9Z]*", base) is not None


def select_valid_edf_channels(available_channels, n_channels=N_EEG_CHANNELS):
    """
    从 EDF 的可用通道中自动选择 n_channels 个 EEG 通道。
    不要求和原项目通道同名，只要求数量一致，并排除 A1/A2/X1/X2/X3/CM 等。
    """
    selected = []
    selected_bases = []
    seen_bases = set()

    for ch in available_channels:
        simple = simplify_channel_name(ch)
        simple_upper = simple.upper()

        if any(kw in simple_upper for kw in EXCLUDE_CHANNEL_KEYWORDS):
            continue

        base = base_channel_name(ch)
        if not is_valid_eeg_base(base):
            continue

        # 同一个 base 只取一次，避免 F3-Pz / F3-Ref 这种重复。
        if base in seen_bases:
            continue

        selected.append(ch)
        selected_bases.append(base)
        seen_bases.add(base)

        if len(selected) >= n_channels:
            break

    if len(selected) < n_channels:
        available_bases = [base_channel_name(ch) for ch in available_channels]
        valid_bases = [b for b in available_bases if is_valid_eeg_base(b)]
        raise ValueError(
            f"可用 EEG 通道不足：需要 {n_channels} 个，实际只选到 {len(selected)} 个。\n"
            f"已选通道：{selected}\n"
            f"可用 base 通道：{available_bases}\n"
            f"有效 base 通道：{valid_bases}\n"
            f"排除 base：{sorted(EXCLUDE_CHANNEL_BASES)}"
        )

    return selected, selected_bases


def get_original_selected_channels(n_channels=N_EEG_CHANNELS):
    """
    原项目 MAT 默认优先使用原论文的 7 个通道。
    如果用户把 N_EEG_CHANNELS 改大，再从 MAT 剩余有效通道里补齐。
    """
    selected = []

    for ch in ORIGINAL_PAPER_CHANNELS:
        if ch in ORIGINAL_MAT_ALL_CHANNELS and is_valid_eeg_base(ch):
            selected.append(ch)
        if len(selected) >= n_channels:
            return selected

    for ch in ORIGINAL_MAT_ALL_CHANNELS:
        if ch not in selected and is_valid_eeg_base(ch):
            selected.append(ch)
        if len(selected) >= n_channels:
            return selected

    raise ValueError(
        f"原项目 MAT 可用通道不足：需要 {n_channels} 个，实际只选到 {len(selected)} 个。"
    )


def split_focus_unfocus(data, order):
    """
    data shape = (n_samples, n_channels)

    order:
    - 'focus_first'：前 10 分钟专注，后 10 分钟不专注
    - 'unfocus_first'：前 10 分钟不专注，后 10 分钟专注
    """
    data = data[:REQUIRED_N, :]
    first = data[:SEGMENT_N, :]
    second = data[SEGMENT_N:REQUIRED_N, :]

    if order == "focus_first":
        focus_raw = first
        unfocus_raw = second
    elif order == "unfocus_first":
        unfocus_raw = first
        focus_raw = second
    else:
        raise ValueError("order 必须是 'focus_first' 或 'unfocus_first'")

    return focus_raw, unfocus_raw


def butter_bandpass_filter(data_1d, lowcut=LOWCUT, highcut=HIGHCUT, fs=TARGET_FS, order=FILTER_ORDER):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq

    if not (0 < low < high < 1):
        raise ValueError(f"滤波频率设置错误：lowcut={lowcut}, highcut={highcut}, fs={fs}")

    b, a = signal.butter(order, [low, high], btype="band")
    return signal.filtfilt(b, a, data_1d)


def remove_baseline_1d(x):
    """
    优先使用原项目 BaselineRemoval.IModPoly()。
    如果没装 BaselineRemoval，就退回到减均值。
    """
    x = np.asarray(x, dtype=float)
    if HAS_BASELINE_REMOVAL:
        try:
            return np.asarray(BaselineRemoval(x).IModPoly(), dtype=float)
        except Exception:
            pass
    return x - np.mean(x)


def preprocess_like_original(raw_data, fs=TARGET_FS):
    """
    输入 raw_data: (n_samples, n_channels)
    输出 filtered_data: (n_samples, n_channels)

    处理流程尽量贴近原项目：
    每通道去基线 / 去均值 -> Butterworth 0.2-43Hz 带通滤波
    """
    filtered_data = np.zeros_like(raw_data, dtype=float)

    for col in range(raw_data.shape[1]):
        detrended = remove_baseline_1d(raw_data[:, col])
        filtered_data[:, col] = butter_bandpass_filter(
            detrended,
            lowcut=LOWCUT,
            highcut=HIGHCUT,
            fs=fs,
            order=FILTER_ORDER,
        )

    return filtered_data

In [4]:
# =========================
# 3. 原项目风格特征提取
# =========================

def extract_original_style_features(filtered_data, fs=TARGET_FS):
    """
    输入：
    filtered_data shape = (10min_samples, n_channels)

    输出：
    X_segment shape = (585, n_channels * 36)

    n_channels * 36 = 通道数 * 36 个频率 bin
    585 = 原项目时间窗口数量
    """
    channel_windows = []

    for col in range(filtered_data.shape[1]):
        f, t, Zxx = signal.stft(
            filtered_data[:, col],
            fs=fs,
            window=signal.windows.blackman(STFT_NPERSEG),
            nperseg=STFT_NPERSEG,
            nfft=STFT_NFFT,
            noverlap=STFT_NOVERLAP,
        )

        power = np.abs(Zxx) ** 2  # shape = (513, 601)

        # 频率 bin：原项目 for i in range(1, 144, 4)
        freq_bins = []
        for i in range(FREQ_BIN_START, FREQ_BIN_STOP, FREQ_BIN_STEP):
            freq_bins.append(np.average(power[i:i + FREQ_BIN_STEP, :], axis=0))
        power_bin = np.stack(freq_bins, axis=0)  # shape = (36, time_windows)

        # 时间窗口：原项目 for i in range(585)
        if power_bin.shape[1] < ORIGINAL_WINDOW_COUNT + ORIGINAL_TIME_AVG_WIDTH:
            raise ValueError(
                f"STFT 时间窗不足：当前 {power_bin.shape[1]}，"
                f"至少需要 {ORIGINAL_WINDOW_COUNT + ORIGINAL_TIME_AVG_WIDTH}"
            )

        power_win = np.zeros((power_bin.shape[0], ORIGINAL_WINDOW_COUNT))
        for i in range(ORIGINAL_WINDOW_COUNT):
            power_win[:, i] = np.average(
                power_bin[:, i:i + ORIGINAL_TIME_AVG_WIDTH],
                axis=1,
            )

        channel_windows.append(power_win)

    # channel_windows: n_channels 个 (36, 585)
    # -> (n_channels, 36, 585)
    arr = np.stack(channel_windows, axis=0)

    # -> (585, n_channels, 36) -> (585, n_channels * 36)
    X_segment = arr.transpose(2, 0, 1).reshape(ORIGINAL_WINDOW_COUNT, -1)

    # 避免 log10(0)
    X_segment = 10 * np.log10(np.maximum(X_segment, 1e-12))

    return X_segment

In [5]:
# =========================
# 4. 读取自采 EDF 数据
# =========================

def get_self_edf_plan():
    edf_files = sorted(SELF_EDF_DIR.glob("*.edf"))

    if SELF_FILE_TABLE is not None:
        table = SELF_FILE_TABLE.copy()
        table["path"] = table["filename"].apply(lambda x: SELF_EDF_DIR / x)
        table["recordname"] = table["path"].apply(lambda p: p.stem)
        table["file_group"] = ["self_file_%02d" % i for i in range(len(table))]
        table["subject_group"] = table["subject_id"].apply(lambda x: f"self_{x}")
        return table

    rows = []
    for i, path in enumerate(edf_files):
        subject_idx = i // FILES_PER_SELF_SUBJECT + 1
        session_idx = i % FILES_PER_SELF_SUBJECT + 1

        rows.append({
            "filename": path.name,
            "path": path,
            "recordname": path.stem,
            "subject_id": f"S{subject_idx:02d}",
            "session_id": session_idx,
            "file_group": f"self_file_{i:02d}",
            "subject_group": f"self_S{subject_idx:02d}",
        })

    return pd.DataFrame(rows)


def load_self_edf_selected_channels(edf_path, target_fs=TARGET_FS):
    raw = mne.io.read_raw_edf(
        str(edf_path),
        preload=True,
        infer_types=True,
        verbose=False,
    )

    rename_dict = {ch: simplify_channel_name(ch) for ch in raw.ch_names}
    raw.rename_channels(rename_dict)

    # 优先只保留 EEG；如果 EDF 没有正确标 EEG 类型，就退回全部通道再筛选。
    try:
        eeg_raw = raw.copy().pick("eeg")
        if len(eeg_raw.ch_names) == 0:
            eeg_raw = raw.copy()
    except Exception:
        eeg_raw = raw.copy()

    available = eeg_raw.ch_names
    available_bases = [base_channel_name(ch) for ch in available]

    selected_channels, selected_bases = select_valid_edf_channels(
        available,
        n_channels=N_EEG_CHANNELS,
    )

    print("  EDF可用base通道:", available_bases)
    print("  自动排除base通道:", sorted(EXCLUDE_CHANNEL_BASES))
    print(f"  本次自动选择前 {N_EEG_CHANNELS} 个有效EEG通道:", selected_channels)
    print("  本次使用base通道:", selected_bases)

    eeg_raw.pick(selected_channels)

    if int(round(eeg_raw.info["sfreq"])) != target_fs:
        eeg_raw.resample(target_fs, npad="auto")

    fs = int(round(eeg_raw.info["sfreq"]))

    # MNE EEG 通常单位是 V，转成 μV
    data_uv = eeg_raw.get_data().T * 1e6

    return data_uv, fs, selected_channels, selected_bases


def build_self_dataset():
    plan_df = get_self_edf_plan()

    print("自采 EDF 文件夹:", SELF_EDF_DIR)
    print("找到自采 EDF 数量:", len(plan_df))

    if len(plan_df) == 0:
        print("没有找到自采 EDF。请检查 SELF_EDF_DIR。")
        return None, plan_df

    display(plan_df[["filename", "subject_id", "session_id", "file_group", "subject_group"]])

    X_list, y_list = [], []
    file_groups, subject_groups, recordnames = [], [], []
    meta_rows = []

    for idx, row in plan_df.iterrows():
        path = row["path"]
        recordname = row["recordname"]

        print("\n读取自采 EDF:", recordname)

        if not Path(path).exists():
            print("  文件不存在，跳过:", path)
            continue

        try:
            data, fs, matched_channels, matched_bases = load_self_edf_selected_channels(path, target_fs=TARGET_FS)
        except Exception as e:
            print("  读取失败，跳过:", recordname)
            print("  错误信息:", e)
            continue

        if data.shape[0] < REQUIRED_N:
            print("  数据长度不足 20 分钟，跳过。")
            print("  当前采样点:", data.shape[0], "需要:", REQUIRED_N)
            continue

        focus_raw, unfocus_raw = split_focus_unfocus(data, order=SELF_SEGMENT_ORDER)

        focus_filtered = preprocess_like_original(focus_raw, fs=fs)
        unfocus_filtered = preprocess_like_original(unfocus_raw, fs=fs)

        X_focus = extract_original_style_features(focus_filtered, fs=fs)
        X_unfocus = extract_original_style_features(unfocus_filtered, fs=fs)

        # 标签固定：focus = 1, unfocus = 0
        y_focus = np.ones(X_focus.shape[0], dtype=int)
        y_unfocus = np.zeros(X_unfocus.shape[0], dtype=int)

        X_record = np.concatenate([X_focus, X_unfocus], axis=0)
        y_record = np.concatenate([y_focus, y_unfocus], axis=0)

        X_list.append(X_record)
        y_list.append(y_record)

        file_groups.extend([row["file_group"]] * len(y_record))
        subject_groups.extend([row["subject_group"]] * len(y_record))
        recordnames.extend([recordname] * len(y_record))

        meta_rows.append({
            "domain": "self_edf",
            "recordname": recordname,
            "filename": Path(path).name,
            "subject_id": row["subject_id"],
            "session_id": row["session_id"],
            "file_group": row["file_group"],
            "subject_group": row["subject_group"],
            "fs": fs,
            "matched_channels": matched_channels,
            "matched_bases": matched_bases,
            "X_focus_shape": X_focus.shape,
            "X_unfocus_shape": X_unfocus.shape,
            "order": SELF_SEGMENT_ORDER,
        })

        print("  matched_channels:", matched_channels)
        print("  matched_bases:", matched_bases)
        print("  X_focus:", X_focus.shape, "X_unfocus:", X_unfocus.shape)

    if len(X_list) == 0:
        print("没有成功构造自采数据集。")
        return None, plan_df

    dataset = {
        "name": "self_edf",
        "X": np.concatenate(X_list, axis=0),
        "y": np.concatenate(y_list, axis=0),
        "file_groups": np.array(file_groups),
        "subject_groups": np.array(subject_groups),
        "recordnames": np.array(recordnames),
        "meta": pd.DataFrame(meta_rows),
    }

    print("\n========== 自采数据集 ==========")
    print("X:", dataset["X"].shape)
    print("y:", dataset["y"].shape)
    print("标签分布:", dict(zip(*np.unique(dataset["y"], return_counts=True))))
    print("文件数:", len(np.unique(dataset["file_groups"])))
    print("被试数:", len(np.unique(dataset["subject_groups"])))

    return dataset, plan_df


self_ds, self_plan_df = build_self_dataset()

自采 EDF 文件夹: newdata\formal_20min
找到自采 EDF 数量: 9


,filename,subject_id,session_id,file_group,subject_group
0,data_0002_raw.edf,S01,1,self_file_00,self_S01
1,data_0003_raw.edf,S01,2,self_file_01,self_S01
2,data_0004_raw.edf,S01,3,self_file_02,self_S01
3,data_zqd_1_raw.edf,S02,1,self_file_03,self_S02
4,data_zqd_2_raw.edf,S02,2,self_file_04,self_S02
5,data_zqd_3_raw.edf,S02,3,self_file_05,self_S02
6,data_zyf_1_raw.edf,S03,1,self_file_06,self_S03
7,data_zyf_2_raw.edf,S03,2,self_file_07,self_S03
8,data_zyf_3_raw.edf,S03,3,self_file_08,self_S03



读取自采 EDF: data_0002_raw
  EDF可用base通道: ['P3', 'C3', 'F3', 'FZ', 'F4', 'C4', 'P4', 'CZ', 'CM', 'A1', 'FP1', 'FP2', 'T3', 'T5', 'O1', 'O2', 'X3', 'X2', 'F7', 'F8', 'X1', 'A2', 'T6', 'T4']
  自动排除base通道: ['A1', 'A2', 'CM', 'X1', 'X2', 'X3']
  本次自动选择前 7 个有效EEG通道: ['P3-Pz', 'C3-Pz', 'F3-Pz', 'Fz-Pz', 'F4-Pz', 'C4-Pz', 'P4-Pz']
  本次使用base通道: ['P3', 'C3', 'F3', 'FZ', 'F4', 'C4', 'P4']
  matched_channels: ['P3-Pz', 'C3-Pz', 'F3-Pz', 'Fz-Pz', 'F4-Pz', 'C4-Pz', 'P4-Pz']
  matched_bases: ['P3', 'C3', 'F3', 'FZ', 'F4', 'C4', 'P4']
  X_focus: (585, 252) X_unfocus: (585, 252)

读取自采 EDF: data_0003_raw
  EDF可用base通道: ['P3', 'C3', 'F3', 'FZ', 'F4', 'C4', 'P4', 'CZ', 'CM', 'A1', 'FP1', 'FP2', 'T3', 'T5', 'O1', 'O2', 'X3', 'X2', 'F7', 'F8', 'X1', 'A2', 'T6', 'T4']
  自动排除base通道: ['A1', 'A2', 'CM', 'X1', 'X2', 'X3']
  本次自动选择前 7 个有效EEG通道: ['P3-Pz', 'C3-Pz', 'F3-Pz', 'Fz-Pz', 'F4-Pz', 'C4-Pz', 'P4-Pz']
  本次使用base通道: ['P3', 'C3', 'F3', 'FZ', 'F4', 'C4', 'P4']
  matched_channels: ['P3-Pz', 'C3-Pz', 'F3-Pz', 'F

In [6]:
# =========================
# 5. 读取原项目 MAT 数据
# =========================

def get_original_mat_plan():
    """
    复刻原 notebook 的循环逻辑。

    原代码：
    for person in range(1, 6):
        if person == 5:
            a = 7
        for day in range(1, a):
            if day == 1 or day == 2: continue
            if person == 4 and day == 7: continue
            recordname = 'eeg_record' + str(7 * (person - 1) + day)
    """
    rows = []
    a = 8

    for person in range(1, 6):
        if person == 5:
            a = 7

        for day in range(1, a):
            if day == 1 or day == 2:
                continue
            if person == 4 and day == 7:
                continue

            record_num = 7 * (person - 1) + day
            recordname = "eeg_record" + str(record_num)
            path = ORIGINAL_MAT_DIR / f"{recordname}.mat"

            rows.append({
                "filename": path.name,
                "path": path,
                "recordname": recordname,
                "subject_id": f"P{person:02d}",
                "session_id": day,
                "file_group": f"original_file_{record_num:02d}",
                "subject_group": f"original_P{person:02d}",
            })

    return pd.DataFrame(rows)


def load_original_mat_selected_channels(mat_path):
    data = scipy.io.loadmat(mat_path)
    data = data["o"]["data"][0][0]

    selected_channels = get_original_selected_channels(N_EEG_CHANNELS)

    eegraw = data[:REQUIRED_N, 3:17]
    eegraw = pd.DataFrame(eegraw, columns=ORIGINAL_MAT_ALL_CHANNELS)
    eegraw = eegraw[selected_channels]

    return eegraw.values.astype(float), TARGET_FS, selected_channels


def build_original_dataset():
    plan_df = get_original_mat_plan()

    print("原项目 MAT 文件夹:", ORIGINAL_MAT_DIR)
    print("计划读取 MAT 数量:", len(plan_df))

    missing = [row["filename"] for _, row in plan_df.iterrows() if not Path(row["path"]).exists()]
    if len(missing) > 0:
        print("缺失 MAT 文件数量:", len(missing))
        print("前几个缺失文件:", missing[:10])

    X_list, y_list = [], []
    file_groups, subject_groups, recordnames = [], [], []
    meta_rows = []

    for idx, row in plan_df.iterrows():
        path = row["path"]
        recordname = row["recordname"]

        if not Path(path).exists():
            continue

        print("\n读取原项目 MAT:", recordname)

        try:
            data, fs, selected_channels = load_original_mat_selected_channels(path)
        except Exception as e:
            print("  读取失败，跳过:", recordname)
            print("  错误信息:", e)
            continue

        if data.shape[0] < REQUIRED_N:
            print("  数据长度不足 20 分钟，跳过。")
            print("  当前采样点:", data.shape[0], "需要:", REQUIRED_N)
            continue

        focus_raw, unfocus_raw = split_focus_unfocus(data, order=ORIGINAL_SEGMENT_ORDER)

        focus_filtered = preprocess_like_original(focus_raw, fs=fs)
        unfocus_filtered = preprocess_like_original(unfocus_raw, fs=fs)

        X_focus = extract_original_style_features(focus_filtered, fs=fs)
        X_unfocus = extract_original_style_features(unfocus_filtered, fs=fs)

        y_focus = np.ones(X_focus.shape[0], dtype=int)
        y_unfocus = np.zeros(X_unfocus.shape[0], dtype=int)

        X_record = np.concatenate([X_focus, X_unfocus], axis=0)
        y_record = np.concatenate([y_focus, y_unfocus], axis=0)

        X_list.append(X_record)
        y_list.append(y_record)

        file_groups.extend([row["file_group"]] * len(y_record))
        subject_groups.extend([row["subject_group"]] * len(y_record))
        recordnames.extend([recordname] * len(y_record))

        meta_rows.append({
            "domain": "original_mat",
            "recordname": recordname,
            "filename": Path(path).name,
            "subject_id": row["subject_id"],
            "session_id": row["session_id"],
            "file_group": row["file_group"],
            "subject_group": row["subject_group"],
            "fs": fs,
            "matched_channels": selected_channels,
            "matched_bases": selected_channels,
            "X_focus_shape": X_focus.shape,
            "X_unfocus_shape": X_unfocus.shape,
            "order": ORIGINAL_SEGMENT_ORDER,
        })

        print("  selected_channels:", selected_channels)
        print("  X_focus:", X_focus.shape, "X_unfocus:", X_unfocus.shape)

    if len(X_list) == 0:
        print("没有成功构造原项目数据集。请检查 ORIGINAL_MAT_DIR。")
        return None, plan_df

    dataset = {
        "name": "original_mat",
        "X": np.concatenate(X_list, axis=0),
        "y": np.concatenate(y_list, axis=0),
        "file_groups": np.array(file_groups),
        "subject_groups": np.array(subject_groups),
        "recordnames": np.array(recordnames),
        "meta": pd.DataFrame(meta_rows),
    }

    print("\n========== 原项目数据集 ==========")
    print("X:", dataset["X"].shape)
    print("y:", dataset["y"].shape)
    print("标签分布:", dict(zip(*np.unique(dataset["y"], return_counts=True))))
    print("文件数:", len(np.unique(dataset["file_groups"])))
    print("被试数:", len(np.unique(dataset["subject_groups"])))

    return dataset, plan_df


original_ds, original_plan_df = build_original_dataset()

原项目 MAT 文件夹: originaldata
计划读取 MAT 数量: 23

读取原项目 MAT: eeg_record3
  selected_channels: ['F7', 'F3', 'P7', 'O1', 'O2', 'P8', 'AF4']
  X_focus: (585, 252) X_unfocus: (585, 252)

读取原项目 MAT: eeg_record4
  selected_channels: ['F7', 'F3', 'P7', 'O1', 'O2', 'P8', 'AF4']
  X_focus: (585, 252) X_unfocus: (585, 252)

读取原项目 MAT: eeg_record5
  selected_channels: ['F7', 'F3', 'P7', 'O1', 'O2', 'P8', 'AF4']
  X_focus: (585, 252) X_unfocus: (585, 252)

读取原项目 MAT: eeg_record6
  selected_channels: ['F7', 'F3', 'P7', 'O1', 'O2', 'P8', 'AF4']
  X_focus: (585, 252) X_unfocus: (585, 252)

读取原项目 MAT: eeg_record7
  selected_channels: ['F7', 'F3', 'P7', 'O1', 'O2', 'P8', 'AF4']
  X_focus: (585, 252) X_unfocus: (585, 252)

读取原项目 MAT: eeg_record10
  selected_channels: ['F7', 'F3', 'P7', 'O1', 'O2', 'P8', 'AF4']
  X_focus: (585, 252) X_unfocus: (585, 252)

读取原项目 MAT: eeg_record11
  selected_channels: ['F7', 'F3', 'P7', 'O1', 'O2', 'P8', 'AF4']
  X_focus: (585, 252) X_unfocus: (585, 252)

读取原项目 MAT: eeg_record12


In [7]:
# =========================
# 6. 检查数据集
# =========================

def describe_dataset(ds):
    if ds is None:
        return pd.DataFrame()

    return pd.DataFrame([{
        "name": ds["name"],
        "X_shape": str(ds["X"].shape),
        "y_shape": str(ds["y"].shape),
        "feature_dim": ds["X"].shape[1],
        "samples": ds["X"].shape[0],
        "focus_count": int(np.sum(ds["y"] == 1)),
        "unfocus_count": int(np.sum(ds["y"] == 0)),
        "file_groups": len(np.unique(ds["file_groups"])),
        "subject_groups": len(np.unique(ds["subject_groups"])),
    }])


dataset_summary = pd.concat(
    [describe_dataset(original_ds), describe_dataset(self_ds)],
    ignore_index=True,
)

display(dataset_summary)

if original_ds is not None and self_ds is not None:
    if original_ds["X"].shape[1] != self_ds["X"].shape[1]:
        raise ValueError(
            f"特征维度不一致：原项目 {original_ds['X'].shape[1]}，自采 {self_ds['X'].shape[1]}。"
            "请确认两边使用了相同的 N_EEG_CHANNELS 和同一套特征提取。"
        )
    else:
        print("特征维度一致，可以进行五种对比实验。注意：当前是“通道数量一致”，不代表通道位置完全一致。")

,name,X_shape,y_shape,feature_dim,samples,focus_count,unfocus_count,file_groups,subject_groups
0,original_mat,"(26910, 252)","(26910,)",252,26910,13455,13455,23,5
1,self_edf,"(10530, 252)","(10530,)",252,10530,5265,5265,9,3


特征维度一致，可以进行五种对比实验。注意：当前是“通道数量一致”，不代表通道位置完全一致。


In [8]:
# =========================
# 7. 模型与评估函数
# =========================

def make_model():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=PCA_N_COMPONENTS, random_state=RANDOM_STATE)),
        ("svm", SVC(kernel="rbf", random_state=RANDOM_STATE, probability=True)),
    ])


def evaluate_once(experiment, X_train, y_train, X_test, y_test, fold_info=""):
    model = make_model()

    start = time.time()
    model.fit(X_train, y_train)
    train_seconds = time.time() - start

    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    row = {
        "experiment": experiment,
        "fold_info": fold_info,
        "train_samples": len(y_train),
        "test_samples": len(y_test),
        "train_focus": int(np.sum(y_train == 1)),
        "train_unfocus": int(np.sum(y_train == 0)),
        "test_focus": int(np.sum(y_test == 1)),
        "test_unfocus": int(np.sum(y_test == 0)),
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision_focus": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall_focus": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_focus": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "tn_unfocus_to_unfocus": int(tn),
        "fp_unfocus_to_focus": int(fp),
        "fn_focus_to_unfocus": int(fn),
        "tp_focus_to_focus": int(tp),
        "train_seconds": train_seconds,
        "pca_components": getattr(model.named_steps["pca"], "n_components_", None),
    }

    return row, model, y_pred


def print_report(title, y_test, y_pred):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    print("混淆矩阵 labels=[unfocus(0), focus(1)]:")
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))
    print("\n分类报告:")
    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["unfocus", "focus"],
        zero_division=0,
    ))


def summarize_results(results_df):
    metric_cols = [
        "accuracy",
        "balanced_accuracy",
        "precision_focus",
        "recall_focus",
        "f1_focus",
        "f1_macro",
    ]

    summary = (
        results_df
        .groupby("experiment")[metric_cols]
        .agg(["mean", "std", "count"])
    )

    return summary

In [9]:
# =========================
# A. 原项目 -> 原项目：按被试留一法
# =========================

def experiment_A_original_to_original(original_ds):
    if original_ds is None:
        print("缺少原项目数据，跳过实验 A。")
        return pd.DataFrame()

    X = original_ds["X"]
    y = original_ds["y"]
    groups = original_ds["subject_groups"]

    logo = LeaveOneGroupOut()
    rows = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups=groups), start=1):
        test_group = np.unique(groups[test_idx])[0]
        row, model, y_pred = evaluate_once(
            experiment="A_original_train_original_test_subject_leave_one",
            X_train=X[train_idx],
            y_train=y[train_idx],
            X_test=X[test_idx],
            y_test=y[test_idx],
            fold_info=f"test_subject={test_group}",
        )
        rows.append(row)
        print(f"A fold {fold}: test_subject={test_group}, acc={row['accuracy']:.4f}, bal_acc={row['balanced_accuracy']:.4f}")

    return pd.DataFrame(rows)


results_A = experiment_A_original_to_original(original_ds)
display(results_A)

A fold 1: test_subject=original_P01, acc=0.6530, bal_acc=0.6530
A fold 2: test_subject=original_P02, acc=0.7822, bal_acc=0.7822
A fold 3: test_subject=original_P03, acc=0.7294, bal_acc=0.7294
A fold 4: test_subject=original_P04, acc=0.6058, bal_acc=0.6058
A fold 5: test_subject=original_P05, acc=0.6122, bal_acc=0.6122


,experiment,fold_info,train_samples,test_samples,train_focus,train_unfocus,test_focus,test_unfocus,accuracy,balanced_accuracy,precision_focus,recall_focus,f1_focus,f1_macro,tn_unfocus_to_unfocus,fp_unfocus_to_focus,fn_focus_to_unfocus,tp_focus_to_focus,train_seconds,pca_components
0,A_original_train_original_test_subject_leave_one,test_subject=original_P01,21060,5850,10530,10530,2925,2925,0.652991,0.652991,0.670087,0.602735,0.634629,0.652113,2057,868,1162,1763,60.009027,29
1,A_original_train_original_test_subject_leave_one,test_subject=original_P02,21060,5850,10530,10530,2925,2925,0.782222,0.782222,0.844676,0.691624,0.760526,0.780420,2553,372,902,2023,50.845125,27
2,A_original_train_original_test_subject_leave_one,test_subject=original_P03,21060,5850,10530,10530,2925,2925,0.729402,0.729402,0.663022,0.932991,0.775174,0.717701,1538,1387,196,2729,65.009878,28
3,A_original_train_original_test_subject_leave_one,test_subject=original_P04,22230,4680,11115,11115,2340,2340,0.605769,0.605769,0.680525,0.398718,0.502829,0.588111,1902,438,1407,933,74.049061,31
4,A_original_train_original_test_subject_leave_one,test_subject=original_P05,22230,4680,11115,11115,2340,2340,0.612179,0.612179,0.574850,0.861538,0.689584,0.586466,849,1491,324,2016,72.288506,29


In [10]:
# =========================
# B. 自采 -> 自采：按被试留一法
# =========================

def experiment_B_self_to_self(self_ds):
    if self_ds is None:
        print("缺少自采数据，跳过实验 B。")
        return pd.DataFrame()

    X = self_ds["X"]
    y = self_ds["y"]
    groups = self_ds["subject_groups"]

    logo = LeaveOneGroupOut()
    rows = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups=groups), start=1):
        test_group = np.unique(groups[test_idx])[0]
        row, model, y_pred = evaluate_once(
            experiment="B_self_train_self_test_subject_leave_one",
            X_train=X[train_idx],
            y_train=y[train_idx],
            X_test=X[test_idx],
            y_test=y[test_idx],
            fold_info=f"test_subject={test_group}",
        )
        rows.append(row)
        print(f"B fold {fold}: test_subject={test_group}, acc={row['accuracy']:.4f}, bal_acc={row['balanced_accuracy']:.4f}")

    return pd.DataFrame(rows)


results_B = experiment_B_self_to_self(self_ds)
display(results_B)

B fold 1: test_subject=self_S01, acc=0.5986, bal_acc=0.5986
B fold 2: test_subject=self_S02, acc=0.6858, bal_acc=0.6858
B fold 3: test_subject=self_S03, acc=0.7516, bal_acc=0.7516


,experiment,fold_info,train_samples,test_samples,train_focus,train_unfocus,test_focus,test_unfocus,accuracy,balanced_accuracy,precision_focus,recall_focus,f1_focus,f1_macro,tn_unfocus_to_unfocus,fp_unfocus_to_focus,fn_focus_to_unfocus,tp_focus_to_focus,train_seconds,pca_components
0,B_self_train_self_test_subject_leave_one,test_subject=self_S01,7020,3510,3510,3510,1755,1755,0.598575,0.598575,0.955263,0.206838,0.340047,0.525806,1738,17,1392,363,3.830283,35
1,B_self_train_self_test_subject_leave_one,test_subject=self_S02,7020,3510,3510,3510,1755,1755,0.685755,0.685755,0.636975,0.863818,0.733253,0.675465,891,864,239,1516,2.733783,42
2,B_self_train_self_test_subject_leave_one,test_subject=self_S03,7020,3510,3510,3510,1755,1755,0.751567,0.751567,0.708353,0.855271,0.774910,0.748866,1137,618,254,1501,3.936526,31


In [11]:
# =========================
# C. 原项目 -> 自采
# =========================

def experiment_C_original_train_self_test(original_ds, self_ds):
    if original_ds is None or self_ds is None:
        print("缺少原项目或自采数据，跳过实验 C。")
        return pd.DataFrame()

    row, model, y_pred = evaluate_once(
        experiment="C_original_train_self_test",
        X_train=original_ds["X"],
        y_train=original_ds["y"],
        X_test=self_ds["X"],
        y_test=self_ds["y"],
        fold_info="train_all_original_test_all_self",
    )

    print_report("实验 C：原项目训练 -> 自采测试", self_ds["y"], y_pred)

    return pd.DataFrame([row])


results_C = experiment_C_original_train_self_test(original_ds, self_ds)
display(results_C)


实验 C：原项目训练 -> 自采测试
混淆矩阵 labels=[unfocus(0), focus(1)]:
[[ 763 4502]
 [1192 4073]]

分类报告:
              precision    recall  f1-score   support

     unfocus       0.39      0.14      0.21      5265
       focus       0.47      0.77      0.59      5265

    accuracy                           0.46     10530
   macro avg       0.43      0.46      0.40     10530
weighted avg       0.43      0.46      0.40     10530



,experiment,fold_info,train_samples,test_samples,train_focus,train_unfocus,test_focus,test_unfocus,accuracy,balanced_accuracy,precision_focus,recall_focus,f1_focus,f1_macro,tn_unfocus_to_unfocus,fp_unfocus_to_focus,fn_focus_to_unfocus,tp_focus_to_focus,train_seconds,pca_components
0,C_original_train_self_test,train_all_original_test_all_self,26910,10530,13455,13455,5265,5265,0.459259,0.459259,0.474985,0.773599,0.588584,0.399971,763,4502,1192,4073,72.49108,29


In [12]:
# =========================
# D. 自采 -> 原项目
# =========================

def experiment_D_self_train_original_test(self_ds, original_ds):
    if self_ds is None or original_ds is None:
        print("缺少自采或原项目数据，跳过实验 D。")
        return pd.DataFrame()

    row, model, y_pred = evaluate_once(
        experiment="D_self_train_original_test",
        X_train=self_ds["X"],
        y_train=self_ds["y"],
        X_test=original_ds["X"],
        y_test=original_ds["y"],
        fold_info="train_all_self_test_all_original",
    )

    print_report("实验 D：自采训练 -> 原项目测试", original_ds["y"], y_pred)

    return pd.DataFrame([row])


results_D = experiment_D_self_train_original_test(self_ds, original_ds)
display(results_D)


实验 D：自采训练 -> 原项目测试
混淆矩阵 labels=[unfocus(0), focus(1)]:
[[11788  1667]
 [11985  1470]]

分类报告:
              precision    recall  f1-score   support

     unfocus       0.50      0.88      0.63     13455
       focus       0.47      0.11      0.18     13455

    accuracy                           0.49     26910
   macro avg       0.48      0.49      0.41     26910
weighted avg       0.48      0.49      0.41     26910



,experiment,fold_info,train_samples,test_samples,train_focus,train_unfocus,test_focus,test_unfocus,accuracy,balanced_accuracy,precision_focus,recall_focus,f1_focus,f1_macro,tn_unfocus_to_unfocus,fp_unfocus_to_focus,fn_focus_to_unfocus,tp_focus_to_focus,train_seconds,pca_components
0,D_self_train_original_test,train_all_self_test_all_original,10530,26910,5265,5265,13455,13455,0.492679,0.492679,0.468601,0.109253,0.177194,0.40524,11788,1667,11985,1470,4.052856,35


In [13]:
# =========================
# E. 原项目 + 部分自采 -> 留出自采被试
# =========================

def experiment_E_mixed_train_self_subject_test(original_ds, self_ds):
    if original_ds is None or self_ds is None:
        print("缺少原项目或自采数据，跳过实验 E。")
        return pd.DataFrame()

    X_self = self_ds["X"]
    y_self = self_ds["y"]
    self_subjects = self_ds["subject_groups"]

    rows = []

    for test_subject in np.unique(self_subjects):
        test_mask = self_subjects == test_subject
        train_self_mask = ~test_mask

        X_train = np.concatenate([
            original_ds["X"],
            X_self[train_self_mask],
        ], axis=0)

        y_train = np.concatenate([
            original_ds["y"],
            y_self[train_self_mask],
        ], axis=0)

        X_test = X_self[test_mask]
        y_test = y_self[test_mask]

        row, model, y_pred = evaluate_once(
            experiment="E_original_plus_self_train_self_subject_test",
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            fold_info=f"test_self_subject={test_subject}",
        )
        rows.append(row)

        print(
            f"E fold test_self_subject={test_subject}: "
            f"acc={row['accuracy']:.4f}, bal_acc={row['balanced_accuracy']:.4f}"
        )

    return pd.DataFrame(rows)


results_E = experiment_E_mixed_train_self_subject_test(original_ds, self_ds)
display(results_E)

E fold test_self_subject=self_S01: acc=0.5066, bal_acc=0.5066
E fold test_self_subject=self_S02: acc=0.7003, bal_acc=0.7003
E fold test_self_subject=self_S03: acc=0.7504, bal_acc=0.7504


,experiment,fold_info,train_samples,test_samples,train_focus,train_unfocus,test_focus,test_unfocus,accuracy,balanced_accuracy,precision_focus,recall_focus,f1_focus,f1_macro,tn_unfocus_to_unfocus,fp_unfocus_to_focus,fn_focus_to_unfocus,tp_focus_to_focus,train_seconds,pca_components
0,E_original_plus_self_train_self_subject_test,test_self_subject=self_S01,33930,3510,16965,16965,1755,1755,0.506553,0.506553,0.509705,0.344160,0.410884,0.493187,1174,581,1151,604,151.462587,28
1,E_original_plus_self_train_self_subject_test,test_self_subject=self_S02,33930,3510,16965,16965,1755,1755,0.700285,0.700285,0.648250,0.875783,0.745032,0.690760,921,834,218,1537,232.633453,29
2,E_original_plus_self_train_self_subject_test,test_self_subject=self_S03,33930,3510,16965,16965,1755,1755,0.750427,0.750427,0.721411,0.815954,0.765775,0.749351,1202,553,323,1432,234.990571,28


In [14]:
# =========================
# 9. 汇总五个实验结果
# =========================

all_results = pd.concat(
    [results_A, results_B, results_C, results_D, results_E],
    ignore_index=True,
)

if len(all_results) == 0:
    print("没有实验结果。请先确认数据路径并运行前面的读取代码。")
else:
    display(all_results)

    print("\n========== 按实验汇总 ==========")
    summary = summarize_results(all_results)
    display(summary)

    out_path = Path("comparison_results_original_vs_self.csv")
    all_results.to_csv(out_path, index=False, encoding="utf-8-sig")
    print("结果已保存:", out_path)

,experiment,fold_info,train_samples,test_samples,train_focus,train_unfocus,test_focus,test_unfocus,accuracy,balanced_accuracy,precision_focus,recall_focus,f1_focus,f1_macro,tn_unfocus_to_unfocus,fp_unfocus_to_focus,fn_focus_to_unfocus,tp_focus_to_focus,train_seconds,pca_components
0,A_original_train_original_test_subject_leave_one,test_subject=original_P01,21060,5850,10530,10530,2925,2925,0.652991,0.652991,0.670087,0.602735,0.634629,0.652113,2057,868,1162,1763,60.009027,29
1,A_original_train_original_test_subject_leave_one,test_subject=original_P02,21060,5850,10530,10530,2925,2925,0.782222,0.782222,0.844676,0.691624,0.760526,0.780420,2553,372,902,2023,50.845125,27
2,A_original_train_original_test_subject_leave_one,test_subject=original_P03,21060,5850,10530,10530,2925,2925,0.729402,0.729402,0.663022,0.932991,0.775174,0.717701,1538,1387,196,2729,65.009878,28
3,A_original_train_original_test_subject_leave_one,test_subject=original_P04,22230,4680,11115,11115,2340,2340,0.605769,0.605769,0.680525,0.398718,0.502829,0.588111,1902,438,1407,933,74.049061,31
4,A_original_train_original_test_subject_leave_one,test_subject=original_P05,22230,4680,11115,11115,2340,2340,0.612179,0.612179,0.574850,0.861538,0.689584,0.586466,849,1491,324,2016,72.288506,29
5,B_self_train_self_test_subject_leave_one,test_subject=self_S01,7020,3510,3510,3510,1755,1755,0.598575,0.598575,0.955263,0.206838,0.340047,0.525806,1738,17,1392,363,3.830283,35
6,B_self_train_self_test_subject_leave_one,test_subject=self_S02,7020,3510,3510,3510,1755,1755,0.685755,0.685755,0.636975,0.863818,0.733253,0.675465,891,864,239,1516,2.733783,42
7,B_self_train_self_test_subject_leave_one,test_subject=self_S03,7020,3510,3510,3510,1755,1755,0.751567,0.751567,0.708353,0.855271,0.774910,0.748866,1137,618,254,1501,3.936526,31
8,C_original_train_self_test,train_all_original_test_all_self,26910,10530,13455,13455,5265,5265,0.459259,0.459259,0.474985,0.773599,0.588584,0.399971,763,4502,1192,4073,72.491080,29
9,D_self_train_original_test,train_all_self_test_all_original,10530,26910,5265,5265,13455,13455,0.492679,0.492679,0.468601,0.109253,0.177194,0.405240,11788,1667,11985,1470,4.052856,35



========== 按实验汇总 ==========


accuracy                  \
                                                      mean       std count   
experiment                                                                   
A_original_train_original_test_subject_leave_one  0.676513  0.076923     5   
B_self_train_self_test_subject_leave_one          0.678632  0.076744     3   
C_original_train_self_test                        0.459259       NaN     1   
D_self_train_original_test                        0.492679       NaN     1   
E_original_plus_self_train_self_subject_test      0.652422  0.128790     3   

                                                 balanced_accuracy            \
                                                              mean       std   
experiment                                                                     
A_original_train_original_test_subject_leave_one          0.676513  0.076923   
B_self_train_self_test_subject_leave_one                  0.678632  0.076744   
C_original_train_self_test                                0.459259       NaN   
D_self_train_original_test                                0.492679       NaN   
E_original_plus_self_train_self_subject_test              0.652422  0.128790   

                                                       precision_focus  \
                                                 count            mean   
experiment                                                               
A_original_train_original_test_subject_leave_one     5        0.686632   
B_self_train_self_test_subject_leave_one             3        0.766864   
C_original_train_self_test                           1        0.474985   
D_self_train_original_test                           1        0.468601   
E_original_plus_self_train_self_subject_test         3        0.626455   

                                                                 recall_focus  \
                                                       std count         mean   
experiment                                                                      
A_original_train_original_test_subject_leave_one  0.097905     5     0.697521   
B_self_train_self_test_subject_leave_one          0.167016     3     0.641975   
C_original_train_self_test                             NaN     1     0.773599   
D_self_train_original_test                             NaN     1     0.109253   
E_original_plus_self_train_self_subject_test      0.107523     3     0.678632   

                                                                  f1_focus  \
                                                       std count      mean   
experiment                                                                   
A_original_train_original_test_subject_leave_one  0.212515     5  0.672549   
B_self_train_self_test_subject_leave_one          0.376865     3  0.616070   
C_original_train_self_test                             NaN     1  0.588584   
D_self_train_original_test                             NaN     1  0.177194   
E_original_plus_self_train_self_subject_test      0.291203     3  0.640564   

                                                                  f1_macro  \
                                                       std count      mean   
experiment                                                                   
A_original_train_original_test_subject_leave_one  0.110459     5  0.664962   
B_self_train_self_test_subject_leave_one          0.239949     3  0.650046   
C_original_train_self_test                             NaN     1  0.399971   
D_self_train_original_test                             NaN     1  0.405240   
E_original_plus_self_train_self_subject_test      0.199178     3  0.644433   

                                                                  
                                                       std count  
experiment                                                        
A_original_train_original_test_subject_leave_one  0.084179     5  
B_self_train_self_test_s

结果已保存: comparison_results_original_vs_self.csv


In [16]:
# =========================
# 10. 可选：自采文件级划分
# =========================

RUN_FILE_LEVEL_SELF_SPLIT = False

if RUN_FILE_LEVEL_SELF_SPLIT and self_ds is not None:
    gss = GroupShuffleSplit(
        n_splits=5,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )

    rows = []
    X = self_ds["X"]
    y = self_ds["y"]
    groups = self_ds["file_groups"]

    for fold, (train_idx, test_idx) in enumerate(gss.split(X, y, groups=groups), start=1):
        test_files = sorted(np.unique(groups[test_idx]).tolist())
        row, model, y_pred = evaluate_once(
            experiment="optional_self_file_group_split",
            X_train=X[train_idx],
            y_train=y[train_idx],
            X_test=X[test_idx],
            y_test=y[test_idx],
            fold_info=f"test_files={test_files}",
        )
        rows.append(row)
        print(f"file-level fold {fold}: acc={row['accuracy']:.4f}, bal_acc={row['balanced_accuracy']:.4f}")

    file_level_results = pd.DataFrame(rows)
    display(file_level_results)
else:
    print("当前没有运行文件级划分。需要时把 RUN_FILE_LEVEL_SELF_SPLIT 改成 True。")

当前没有运行文件级划分。需要时把 RUN_FILE_LEVEL_SELF_SPLIT 改成 True。


In [18]:
# =========================
# A0. 原项目 -> 原项目：随机窗口切分
# 用来对齐原始 notebook 的高准确率
# =========================

from sklearn.model_selection import train_test_split

def experiment_A0_original_random_split(original_ds):
    X = original_ds["X"]
    y = original_ds["y"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    row, model, y_pred = evaluate_once(
        experiment="A0_original_random_window_split",
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        fold_info="random_window_split_like_original_notebook",
    )

    print_report("A0：原项目随机窗口切分", y_test, y_pred)

    return pd.DataFrame([row])

results_A0 = experiment_A0_original_random_split(original_ds)
display(results_A0)



A0：原项目随机窗口切分
混淆矩阵 labels=[unfocus(0), focus(1)]:
[[2494  197]
 [ 172 2519]]

分类报告:
              precision    recall  f1-score   support

     unfocus       0.94      0.93      0.93      2691
       focus       0.93      0.94      0.93      2691

    accuracy                           0.93      5382
   macro avg       0.93      0.93      0.93      5382
weighted avg       0.93      0.93      0.93      5382



,experiment,fold_info,train_samples,test_samples,train_focus,train_unfocus,test_focus,test_unfocus,accuracy,balanced_accuracy,precision_focus,recall_focus,f1_focus,f1_macro,tn_unfocus_to_unfocus,fp_unfocus_to_focus,fn_focus_to_unfocus,tp_focus_to_focus,train_seconds,pca_components
0,A0_original_random_window_split,random_window_split_like_original_notebook,21528,5382,10764,10764,2691,2691,0.931438,0.931438,0.927467,0.936083,0.931755,0.931437,2494,197,172,2519,26.903648,29


In [ ]:
# =========================
# B0. 自采 -> 自采：随机窗口切分
# 用来和原项目随机窗口高准确率做控制变量对比
# =========================

from sklearn.model_selection import train_test_split

def experiment_B0_self_random_window_split(self_ds):
    """
    这个实验和原项目 notebook 的 train_test_split 思路一致：
    把所有自采窗口样本混在一起，随机抽 80% 训练、20% 测试。

    注意：这个结果一般会偏乐观，因为同一个被试、同一个 EDF 文件的相邻窗口
    可能同时出现在训练集和测试集里。它适合用来做“控制变量对比”，
    不适合作为最终实用性指标。
    """
    if self_ds is None:
        print("缺少自采数据，跳过实验 B0。")
        return pd.DataFrame()

    X = self_ds["X"]
    y = self_ds["y"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    row, model, y_pred = evaluate_once(
        experiment="B0_self_train_self_test_random_window_split",
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        fold_info=f"random_window_split_test_size={TEST_SIZE}, random_state={RANDOM_STATE}",
    )

    print_report("B0：自采数据随机窗口切分", y_test, y_pred)
    return pd.DataFrame([row])


results_B0 = experiment_B0_self_random_window_split(self_ds)
display(results_B0)



B0：自采数据随机窗口切分
混淆矩阵 labels=[unfocus(0), focus(1)]:
[[ 979   74]
 [  27 1026]]

分类报告:
              precision    recall  f1-score   support

     unfocus       0.97      0.93      0.95      1053
       focus       0.93      0.97      0.95      1053

    accuracy                           0.95      2106
   macro avg       0.95      0.95      0.95      2106
weighted avg       0.95      0.95      0.95      2106



,experiment,fold_info,train_samples,test_samples,train_focus,train_unfocus,test_focus,test_unfocus,accuracy,balanced_accuracy,precision_focus,recall_focus,f1_focus,f1_macro,tn_unfocus_to_unfocus,fp_unfocus_to_focus,fn_focus_to_unfocus,tp_focus_to_focus,train_seconds,pca_components
0,B0_self_train_self_test_random_window_split,"random_window_split_test_size=0.2, random_stat...",8424,2106,4212,4212,1053,1053,0.952042,0.952042,0.932727,0.974359,0.953089,0.952018,979,74,27,1026,2.762156,35


: 